# Chunk Span Inspector

Interactively verify that `char_start` / `char_end` on every chunk match the source document text.

Runs against **real `.txt` files** from your data directory — no OpenSearch needed.

In [1]:
import sys

from pathlib import Path
from legalrag.core.models import LegalDocumentMetadata, RawDocument
from legalrag.ingestion.chunker import HierarchicalChunker, RecursiveCharacterTextSplitter
from legalrag.ingestion.loader import TxtFileLoader, clean_document_text

print('Imports OK')

Imports OK


## 1. Load a document

In [2]:
# ── Change this path to any .txt file in your data directory ──────────────────
DOC_PATH = Path('data/LegalBenchRAG/corpus/maud/').glob('*.txt')
doc_path = next(DOC_PATH, None)

if doc_path is None:
    raise FileNotFoundError('No .txt files found under data/subset — adjust DOC_PATH above')

print(f'Loading: {doc_path}')
raw_text = doc_path.read_text(encoding='utf-8', errors='replace')
cleaned_text = clean_document_text(raw_text)
print(f'Raw length:     {len(raw_text):,} chars')
print(f'Cleaned length: {len(cleaned_text):,} chars')
print('\n--- First 300 chars of cleaned text ---')
print(repr(cleaned_text[:300]))

Loading: data/LegalBenchRAG/corpus/maud/Soliton_Inc_Abbvie_Inc.txt
Raw length:     286,202 chars
Cleaned length: 285,114 chars

--- First 300 chars of cleaned text ---
'\ufeffExhibit 2.1 \n\n    AGREEMENT AND PLAN OF MERGER \n\nBy and Among \n\nABBVIE INC., \n\nSCOUT MERGER SUB, INC. \n\nand \n\nSOLITON, INC. \n\nDated as of May 8, 2021 \n\n   \n\n________________\n\nTABLE OF CONTENTS         Page    ARTICLE I    \n\n  The Transactions    \n\nSECTION 1.01.   The Merger    2 SECTION 1.02.   Clo'


In [3]:
metadata = LegalDocumentMetadata(source_path=str(doc_path))
doc = RawDocument(metadata=metadata, text=cleaned_text)

## 2. Chunk with HierarchicalChunker

In [4]:
PARENT_SIZE  = 1500
CHILD_SIZE   = 512
CHILD_OVERLAP = 64

chunker = HierarchicalChunker(
    parent_size=PARENT_SIZE,
    child_size=CHILD_SIZE,
    child_overlap=CHILD_OVERLAP,
)
chunks = chunker.chunk(doc)

parents  = [c for c in chunks if c.is_parent]
children = [c for c in chunks if not c.is_parent]
print(f'Total chunks : {len(chunks)}')
print(f'  Parents    : {len(parents)}')
print(f'  Children   : {len(children)}')

Total chunks : 884
  Parents    : 170
  Children   : 714


## 3. Verify offsets

In [5]:
text = doc.text
mismatches = []
for c in chunks:
    expected = text[c.char_start:c.char_end]
    if c.text != expected:
        mismatches.append({
            'chunk_id'  : c.chunk_id,
            'is_parent' : c.is_parent,
            'char_start': c.char_start,
            'char_end'  : c.char_end,
            'stored'    : repr(c.text[:80]),
            'from_doc'  : repr(expected[:80]),
        })

if mismatches:
    print(f'FAILED: {len(mismatches)} offset mismatch(es)')
    for m in mismatches:
        print(m)
else:
    print(f'PASS — all {len(chunks)} chunks match their char spans')

PASS — all 884 chunks match their char spans


In [6]:
# Parent coverage: must tile [0, len(text)) with no gaps
sorted_parents = sorted(parents, key=lambda c: c.char_start)
gaps = []
if sorted_parents[0].char_start != 0:
    gaps.append(f'First parent starts at {sorted_parents[0].char_start}, not 0')
if sorted_parents[-1].char_end != len(text):
    gaps.append(f'Last parent ends at {sorted_parents[-1].char_end}, expected {len(text)}')
for a, b in zip(sorted_parents, sorted_parents[1:]):
    if a.char_end != b.char_start:
        gaps.append(f'Gap/overlap: [{a.char_start}:{a.char_end}] -> [{b.char_start}:{b.char_end}]')

if gaps:
    print('FAILED — parent coverage issues:')
    for g in gaps:
        print(' ', g)
else:
    print(f'PASS — {len(parents)} parents tile [0, {len(text)}) with no gaps')

PASS — 170 parents tile [0, 285114) with no gaps


In [7]:
# Child bounds: every child must sit within its parent
parent_map = {c.chunk_id: c for c in parents}
out_of_bounds = []
for child in children:
    parent = parent_map.get(child.parent_chunk_id)
    if parent is None:
        out_of_bounds.append(f'Child {child.chunk_id} has unknown parent {child.parent_chunk_id}')
    elif not (child.char_start >= parent.char_start and child.char_end <= parent.char_end):
        out_of_bounds.append(
            f'Child [{child.char_start}:{child.char_end}] outside '
            f'parent [{parent.char_start}:{parent.char_end}]'
        )

if out_of_bounds:
    print(f'FAILED — {len(out_of_bounds)} child bound issue(s):')
    for o in out_of_bounds:
        print(' ', o)
else:
    print(f'PASS — all {len(children)} children within their parent bounds')

PASS — all 714 children within their parent bounds


## 4. Browse individual chunks

In [8]:
# Inspect a specific chunk by index
IDX = 0  # change this
c = chunks[IDX]
print(f'chunk_id    : {c.chunk_id}')
print(f'is_parent   : {c.is_parent}')
print(f'char_start  : {c.char_start}')
print(f'char_end    : {c.char_end}')
print(f'len(text)   : {len(c.text)}')
print(f'\n--- chunk.text ---')
print(c.text)
print(f'\n--- doc.text[{c.char_start}:{c.char_end}] ---')
print(text[c.char_start:c.char_end])
print(f'\nMatch: {c.text == text[c.char_start:c.char_end]}')

chunk_id    : 464fbe5c46624f24c0398f20bcdfcf34
is_parent   : True
char_start  : 0
char_end    : 1516
len(text)   : 1516

--- chunk.text ---
﻿Exhibit 2.1 

    AGREEMENT AND PLAN OF MERGER 

By and Among 

ABBVIE INC., 

SCOUT MERGER SUB, INC. 

and 

SOLITON, INC. 

Dated as of May 8, 2021 

   

________________

TABLE OF CONTENTS         Page    ARTICLE I    

  The Transactions    

SECTION 1.01.   The Merger    2 SECTION 1.02.   Closing    2 SECTION 1.03.   Effective Time    2 SECTION 1.04.   Effects of the Merger    2 SECTION 1.05.   Certificate of Incorporation of the Surviving Corporation    2 SECTION 1.06.   Directors and Officers of the Surviving Corporation    3 

  ARTICLE II    

Effect of the Merger on Capital Stock; Exchange of Certificates; Equity-Based Awards    

SECTION 2.01.   Effect on Capital Stock    3 SECTION 2.02.   Exchange of Certificates and Book Entry Shares    4 SECTION 2.03.   Equity-Based Awards    6 SECTION 2.04.   Payments with Respect to Equity-Based A

## 5. Visualise span layout

In [ ]:
# ASCII ruler showing parent/child spans over the first N chars of the document
N = min(len(text), 3000)
ruler = ['.'] * N

for c in sorted(parents, key=lambda x: x.char_start):
    s, e = max(0, c.char_start), min(N, c.char_end)
    for i in range(s, e):
        ruler[i] = 'P'

for c in sorted(children, key=lambda x: x.char_start):
    s, e = max(0, c.char_start), min(N, c.char_end)
    for i in range(s, e):
        if ruler[i] == 'P':
            ruler[i] = 'C'       # child within a parent — good
        elif ruler[i] == 'C':
            pass                 # overlap between siblings — fine, already covered
        else:                    # ruler[i] == '.' — child outside any parent — bug!
            ruler[i] = 'X'

ruler_str = ''.join(ruler)
print(f'Span map (first {N} chars): P=parent-only, C=child-within-parent, .=uncovered, X=child-outside-parent-bug')
for i in range(0, N, 100):
    print(f'{i:4d}: {ruler_str[i:i+100]}')

if 'X' in ruler_str:
    print('\nWARNING: X characters found — children outside parent bounds!')
elif '.' in ruler_str:
    uncovered = [i for i, ch in enumerate(ruler_str) if ch == '.']
    print(f'\nWARNING: {len(uncovered)} uncovered positions — gaps in parent coverage!')
else:
    print('\nAll positions covered correctly.')

## 6. RecursiveCharacterTextSplitter — same checks

In [10]:
rchunker = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
rchunks = rchunker.chunk(doc)
print(f'RecursiveCharacterTextSplitter: {len(rchunks)} chunks')

mismatches = [
    c for c in rchunks
    if c.text != text[c.char_start:c.char_end]
]
if mismatches:
    print(f'FAILED: {len(mismatches)} offset mismatch(es)')
else:
    print(f'PASS — all {len(rchunks)} chunks match their char spans')

RecursiveCharacterTextSplitter: 788 chunks
PASS — all 788 chunks match their char spans


## 7. Ground-truth span alignment (LegalBench-RAG)

Verify that the GT `span` in the benchmark JSON points at the correct text in the raw corpus file,
and then check how well your chunks cover it.

In [13]:
import json
from pathlib import Path

DATA_DIR    = Path('data/LegalBenchRAG')
BENCHMARK   = 'maud'          # contractnli | cuad | maud | privacy_qa
TEST_IDX    = 0               # which test case to inspect
SNIPPET_IDX = 0               # which snippet within that test case

bench = json.load(open(DATA_DIR / f'benchmarks/{BENCHMARK}.json'))
test  = bench['tests'][TEST_IDX]
snip  = test['snippets'][SNIPPET_IDX]

gt_start, gt_end = snip['span']
corpus_file = DATA_DIR / 'corpus' / snip['file_path']
raw = corpus_file.read_text(encoding='utf-8', errors='replace')

print(f'Query        : {test["query"][:120]}')
print(f'Corpus file  : {snip["file_path"]}')
print(f'GT span      : [{gt_start}, {gt_end})  ({gt_end - gt_start} chars)')
print(f'Raw file len : {len(raw):,} chars')
print()
gt_text = raw[gt_start:gt_end]
print('--- GT text from raw[span] ---')
print(repr(gt_text[:300]))

Query        : Consider the Acquisition Agreement between Parent "Magic AcquireCo, Inc." and Target "The Michaels Companies, Inc."; Wha
Corpus file  : maud/The Michaels Companies, Inc._Apollo Global Management, LLC.txt
GT span      : [5284, 5913)  (629 chars)
Raw file len : 356,640 chars

--- GT text from raw[span] ---
'WHEREAS, pursuant to this Agreement, Merger Subsidiary has agreed to commence, and Parent has agreed to cause Merger Subsidiary to commence, a tender offer (as it may be extended and amended from time to time pursuant to this Agreement, the “Offer”) to purchase any (subject to the Minimum Tender Con'


In [16]:
# Chunk the same corpus file (no cleaning — LegalBenchRAG loader uses raw text)
from legalrag.core.models import LegalDocumentMetadata, RawDocument
from legalrag.ingestion.chunker import HierarchicalChunker

meta   = LegalDocumentMetadata(source_path=str(corpus_file), citation=snip['file_path'])
gt_doc = RawDocument(metadata=meta, text=raw)

gt_chunker = HierarchicalChunker(parent_size=2048, child_size=512, child_overlap=64)
gt_chunks  = gt_chunker.chunk(gt_doc)
gt_children = [c for c in gt_chunks if not c.is_parent]

print(f'Chunked into {len(gt_children)} child chunks')

# ── Which chunks overlap the GT span? ─────────────────────────────────────────
def spans_overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]

hitting = [
    c for c in gt_children
    if spans_overlap((c.char_start, c.char_end), (gt_start, gt_end))
]

print(f'Child chunks overlapping GT span [{gt_start}, {gt_end}): {len(hitting)}')
for h in hitting:
    intersection_start = max(h.char_start, gt_start)
    intersection_end   = min(h.char_end,   gt_end)
    intersection_chars = intersection_end - intersection_start
    coverage_pct       = 100 * intersection_chars / (gt_end - gt_start)
    print(f'  [{h.char_start:6d}, {h.char_end:6d})  '
          f'overlap={intersection_chars} chars  '
          f'GT coverage={coverage_pct:.1f}%')

Chunked into 847 child chunks
Child chunks overlapping GT span [5284, 5913): 2
  [  5103,   5615)  overlap=331 chars  GT coverage=52.6%
  [  5551,   6063)  overlap=362 chars  GT coverage=57.6%


In [15]:
# Total char recall: what fraction of GT chars is covered by all hitting chunks combined?
def merge_spans(spans):
    """Merge overlapping [start, end) spans."""
    if not spans:
        return []
    spans = sorted(spans)
    merged = [spans[0]]
    for s, e in spans[1:]:
        if s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged

retrieved_spans = [(max(h.char_start, gt_start), min(h.char_end, gt_end)) for h in hitting]
merged = merge_spans(retrieved_spans)
covered_chars = sum(e - s for s, e in merged)
gt_chars      = gt_end - gt_start

print(f'GT span      : {gt_chars} chars')
print(f'Covered      : {covered_chars} chars')
print(f'Char recall  : {covered_chars / gt_chars:.3f}  ({100*covered_chars/gt_chars:.1f}%)')
if covered_chars < gt_chars:
    print()
    print('Uncovered ranges within GT span:')
    prev = gt_start
    for s, e in merged:
        if s > prev:
            print(f'  [{prev}, {s})  ({s-prev} chars): {repr(raw[prev:s][:80])}')
        prev = e
    if prev < gt_end:
        print(f'  [{prev}, {gt_end})  ({gt_end-prev} chars): {repr(raw[prev:gt_end][:80])}')

GT span      : 629 chars
Covered      : 629 chars
Char recall  : 1.000  (100.0%)
